<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 5 — Force–Distance Curve Analysis: Python Exercises

**SPM Syllabus 2026** · Interactive simulations for AFM force spectroscopy

These exercises accompany Chapter 5 of the textbook. Each exercise builds an interactive simulation that lets you explore the physics of AFM force–distance curves, from raw signal generation through parameter extraction to contact mechanics fitting.

**How to use this notebook:**
1. Run the first two cells (imports + widgets) before anything else.
2. Each exercise has a *theory cell* (markdown) followed by an *interactive code cell*.
3. Use the sliders to explore — the goal is to build physical intuition, not just run code.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.integrate import trapezoid

# Physical constants
k_B = 1.381e-23   # Boltzmann constant (J/K)
e   = 1.602e-19   # elementary charge (C)

# Plotting defaults
plt.rcParams.update({
    'figure.dpi': 100,
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 9,
    'lines.linewidth': 2,
})

In [ ]:
try:
    from ipywidgets import interact, FloatSlider, IntSlider, FloatLogSlider, Dropdown
except ImportError:
    !pip install ipywidgets -q
    from ipywidgets import interact, FloatSlider, IntSlider, FloatLogSlider, Dropdown

---
## 1. Force–Distance Curve: Approach and Retraction

A force–distance curve records how the interaction force evolves as the AFM tip approaches and then retracts from the surface. The measurement passes through several characteristic **regimes** (Section 5.1):

| Regime | Phase | Physical origin |
|--------|-------|-----------------|
| I | Approach | No interaction (baseline) |
| II | Approach | Attractive van der Waals / electrostatic |
| III | Approach | Jump-to-contact (mechanical instability) |
| IV | Approach / Retraction | Repulsive contact |
| V–VI | Retraction | Adhesion + pull-off |

The **hysteresis** between approach and retraction encodes energy dissipation (Section 5.3.3).

**Key insight:** The approach and retraction curves are generally *not* identical. Their difference is a direct measure of non-conservative interactions (adhesion, viscoelasticity, capillary forces).

### Tasks
1. Generate a synthetic force–distance curve with all regimes.
2. Identify the jump-to-contact, contact point, and pull-off.
3. Vary the cantilever stiffness and adhesion energy to see how the curve shape changes.

In [ ]:
def interactive_fd_curve(k_N_per_m=0.5, F_adh_nN=2.0, E_star_kPa=50, R_tip_nm=20):
    '''
    Simulate a complete approach–retraction force–distance curve.
    '''
    R = R_tip_nm * 1e-9
    E_star = E_star_kPa * 1e3
    k = k_N_per_m

    # Distance array (nm) — approach then retraction
    z_app = np.linspace(50, -20, 1000)   # approach: far → close
    z_ret = np.linspace(-20, 50, 1000)   # retraction: close → far

    def force_approach(z):
        F = np.zeros_like(z)
        # Region I: no interaction (z > 5 nm)
        # Region II: attractive van der Waals  F ~ -H*R / (6*z^2)
        attractive = (z > 0) & (z < 10)
        F[attractive] = -F_adh_nN * 0.3 * np.exp(-z[attractive] / 2.0)
        # Region III: jump-to-contact — check instability
        # dF/dz > k  → snap-in
        jtc_mask = z <= 0
        # Region IV: repulsive Hertz-like contact
        indent = np.abs(z[jtc_mask])
        F[jtc_mask] = (4/3) * E_star * np.sqrt(R) * (indent * 1e-9)**1.5 * 1e9  # nN
        return F

    def force_retraction(z):
        F = np.zeros_like(z)
        # Contact region
        contact = z <= 0
        indent = np.abs(z[contact])
        F[contact] = (4/3) * E_star * np.sqrt(R) * (indent * 1e-9)**1.5 * 1e9
        F[contact] -= F_adh_nN * 0.5  # adhesion offset in contact
        # Adhesion region: tip sticks after z=0
        adh_region = (z > 0) & (z < F_adh_nN / k)
        F[adh_region] = -k * z[adh_region]  # cantilever restoring
        # Pull-off snap
        free = z >= F_adh_nN / k
        F[free] = 0
        return F

    F_app = force_approach(z_app)
    F_ret = force_retraction(z_ret)

    # Pull-off distance
    z_pulloff = F_adh_nN / k

    print(f"  Force–Distance Curve Simulation")
    print(f"  ─────────────────────────────────")
    print(f"  Cantilever stiffness  k     = {k_N_per_m:.2f} N/m")
    print(f"  Adhesion force        F_adh = {F_adh_nN:.2f} nN")
    print(f"  Effective modulus      E*    = {E_star_kPa:.0f} kPa")
    print(f"  Tip radius            R     = {R_tip_nm:.0f} nm")
    print(f"  Pull-off distance     z_po  = {z_pulloff:.1f} nm")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Full F-d curve
    axes[0].plot(z_app, F_app, 'b-', lw=2, label='Approach')
    axes[0].plot(z_ret, F_ret, 'r--', lw=2, label='Retraction')
    axes[0].axhline(0, color='gray', ls=':', lw=0.8)
    axes[0].axvline(0, color='gray', ls=':', lw=0.8, alpha=0.5)
    axes[0].fill_between(z_ret, F_ret, 0, where=(F_ret < 0),
                         alpha=0.15, color='red', label='Adhesion region')
    axes[0].set_xlabel('Tip–sample distance (nm)')
    axes[0].set_ylabel('Force (nN)')
    axes[0].set_title('Force–distance curve')
    axes[0].legend(fontsize=8)
    axes[0].set_xlim(-20, 50)

    # Annotations
    axes[0].annotate('Contact\npoint', xy=(0, 0), xytext=(10, 3),
                     fontsize=8, arrowprops=dict(arrowstyle='->', color='k'))
    if z_pulloff < 50:
        axes[0].annotate('Pull-off', xy=(z_pulloff, -F_adh_nN*0.1),
                         xytext=(z_pulloff+5, -F_adh_nN*0.8), fontsize=8,
                         arrowprops=dict(arrowstyle='->', color='red'))

    # Zoomed contact region
    z_zoom = np.linspace(-20, 5, 500)
    F_app_zoom = force_approach(z_zoom)
    F_ret_zoom = force_retraction(z_zoom)
    axes[1].plot(z_zoom, F_app_zoom, 'b-', lw=2, label='Approach')
    axes[1].plot(z_zoom, F_ret_zoom, 'r--', lw=2, label='Retraction')
    axes[1].axhline(0, color='gray', ls=':', lw=0.8)
    axes[1].axvline(0, color='gray', ls=':', lw=0.8, alpha=0.5)
    axes[1].fill_between(z_zoom, F_app_zoom, F_ret_zoom,
                         alpha=0.1, color='green', label='Hysteresis (W_diss)')
    axes[1].set_xlabel('Tip–sample distance (nm)')
    axes[1].set_ylabel('Force (nN)')
    axes[1].set_title('Contact region (zoomed)')
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

interact(
    interactive_fd_curve,
    k_N_per_m=FloatLogSlider(value=0.5, base=10, min=-2, max=1, step=0.1, description='k (N/m)'),
    F_adh_nN=FloatSlider(value=2.0, min=0.1, max=10, step=0.1, description='F_adh (nN)'),
    E_star_kPa=FloatSlider(value=50, min=1, max=500, step=5, description='E* (kPa)'),
    R_tip_nm=FloatSlider(value=20, min=5, max=100, step=5, description='R_tip (nm)')
);

---
## 2. From Raw Signals to Force–Distance Curve (The AFM Measurement Chain)

The AFM does **not** directly measure force vs distance. It measures (Section 5.2):
- **z-piezo displacement** $z_{\mathrm{piezo}}$ (controlled)
- **Photodiode voltage** $V$ (measured)

The conversion chain is:

$$V \;\xrightarrow{\times\, S}\; d \;\xrightarrow{\times\, k}\; F$$

where $S$ is the deflection sensitivity (nm/V) and $k$ is the spring constant (N/m).

The **true tip–sample separation** requires a geometry correction (Section 5.5):

$$\delta = z_{\mathrm{piezo}} - d$$

where $\delta$ is the indentation into the sample.

**Key insight:** What the AFM records is $(z_{\mathrm{piezo}},\, V)$. Converting this to $(d,\, F)$ or $(\delta,\, F)$ requires calibration — and each step introduces potential errors.

### Tasks
1. Start from raw photodiode voltage and z-piezo data.
2. Apply the measurement chain to convert to force and distance.
3. Explore how errors in sensitivity or spring constant propagate.

In [ ]:
def interactive_measurement_chain(S_nm_per_V=50.0, k_N_per_m=0.3, S_error_pct=0, k_error_pct=0):
    '''
    Simulate the AFM measurement chain: V → d → F, z_piezo → separation.
    '''
    # Generate synthetic raw data (z_piezo in nm, V in volts)
    z_piezo = np.linspace(200, -50, 800)  # nm, approach
    k_true = 0.3   # true spring constant
    S_true = 50.0   # true sensitivity

    # True physics: Hertz contact for z < 0, no interaction for z > 5
    d_true = np.zeros_like(z_piezo)
    contact = z_piezo < 0
    d_true[contact] = 0.7 * np.abs(z_piezo[contact])  # cantilever bends
    attractive = (z_piezo >= 0) & (z_piezo < 10)
    d_true[attractive] = -0.3 * np.exp(-z_piezo[attractive] / 3)

    # Raw voltage signal
    V_raw = d_true / S_true + 0.005 * np.random.default_rng(42).normal(size=len(z_piezo))

    # Apply user's calibration (possibly with errors)
    S_used = S_nm_per_V * (1 + S_error_pct / 100)
    k_used = k_N_per_m * (1 + k_error_pct / 100)

    # Conversion chain
    d_measured = V_raw * S_used          # deflection (nm)
    F_measured = k_used * d_measured      # force (nN)
    separation = z_piezo - d_measured     # true separation (nm)
    indentation = -separation
    indentation[indentation < 0] = 0

    # Reference (perfect calibration)
    d_ref = V_raw * S_true
    F_ref = k_true * d_ref

    print(f"  AFM Measurement Chain")
    print(f"  ─────────────────────────────────")
    print(f"  Sensitivity (set)   S = {S_nm_per_V:.1f} nm/V  (error: {S_error_pct:+.0f}%)")
    print(f"  Spring constant (set) k = {k_N_per_m:.3f} N/m  (error: {k_error_pct:+.0f}%)")
    print(f"  Effective S used      = {S_used:.1f} nm/V")
    print(f"  Effective k used      = {k_used:.3f} N/m")
    print()
    print(f"  Max force (measured)  = {np.max(F_measured):.2f} nN")
    print(f"  Max force (reference) = {np.max(F_ref):.2f} nN")
    print(f"  Force error           = {(np.max(F_measured)/np.max(F_ref) - 1)*100:+.1f}%")

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    # Raw signal
    axes[0, 0].plot(z_piezo, V_raw, 'k-', lw=1.5)
    axes[0, 0].set_xlabel('z-piezo (nm)')
    axes[0, 0].set_ylabel('Photodiode voltage (V)')
    axes[0, 0].set_title('Step 1: Raw signal V(z_piezo)')
    axes[0, 0].axhline(0, color='gray', ls=':', lw=0.8)

    # Deflection
    axes[0, 1].plot(z_piezo, d_measured, 'b-', lw=2, label='Measured')
    axes[0, 1].plot(z_piezo, d_ref, 'k--', lw=1, alpha=0.5, label='Reference')
    axes[0, 1].set_xlabel('z-piezo (nm)')
    axes[0, 1].set_ylabel('Deflection d (nm)')
    axes[0, 1].set_title('Step 2: d = V × S')
    axes[0, 1].legend(fontsize=8)

    # Force vs z-piezo
    axes[1, 0].plot(z_piezo, F_measured, 'b-', lw=2, label='Measured')
    axes[1, 0].plot(z_piezo, F_ref, 'k--', lw=1, alpha=0.5, label='Reference')
    axes[1, 0].set_xlabel('z-piezo (nm)')
    axes[1, 0].set_ylabel('Force F (nN)')
    axes[1, 0].set_title('Step 3: F = k × d')
    axes[1, 0].legend(fontsize=8)

    # Force vs indentation (geometry-corrected)
    indent_ref = z_piezo - d_ref
    axes[1, 1].plot(-separation, F_measured, 'b-', lw=2, label='F(δ) measured')
    axes[1, 1].plot(-indent_ref, F_ref, 'k--', lw=1, alpha=0.5, label='F(δ) reference')
    axes[1, 1].set_xlabel('Indentation δ (nm)')
    axes[1, 1].set_ylabel('Force F (nN)')
    axes[1, 1].set_title('Step 4: δ = z_piezo − d (geometry correction)')
    axes[1, 1].set_xlim(-10, 50)
    axes[1, 1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

interact(
    interactive_measurement_chain,
    S_nm_per_V=FloatSlider(value=50, min=20, max=100, step=1, description='S (nm/V)'),
    k_N_per_m=FloatLogSlider(value=0.3, base=10, min=-2, max=1, step=0.1, description='k (N/m)'),
    S_error_pct=FloatSlider(value=0, min=-30, max=30, step=1, description='S error (%)'),
    k_error_pct=FloatSlider(value=0, min=-30, max=30, step=1, description='k error (%)')
);

---
## 3. Hysteresis and Energy Dissipation

The area enclosed between approach and retraction curves represents the **energy dissipated** during one interaction cycle (Section 5.3.3):

$$W_{\mathrm{diss}} = \oint F \cdot \mathrm{d}d = \int (F_{\mathrm{ret}} - F_{\mathrm{app}}) \, \mathrm{d}d$$

Dissipation arises from adhesion, viscoelasticity, plastic deformation, or capillary effects. In practice, we compute this area using the **trapezoidal rule** on discrete data points.

**Key insight:** The hysteresis area is *not* just a graphical feature — it has a direct physical meaning as mechanical energy lost per cycle. A purely elastic interaction would show zero hysteresis.

### Tasks
1. Simulate approach/retraction with adjustable adhesion and viscoelastic loss.
2. Compute dissipated energy numerically (trapezoidal rule).
3. Compare conservative vs dissipative interactions.

In [ ]:
def interactive_hysteresis(F_adh_nN=2.0, visc_loss_pct=15, max_indent_nm=30):
    '''
    Compute energy dissipation from hysteresis between approach and retraction.
    '''
    # Indentation array (nm)
    delta = np.linspace(0, max_indent_nm, 500)

    # Approach: Hertz-like F = a * delta^1.5
    a = 0.01  # prefactor (nN/nm^1.5)
    F_app = a * delta**1.5

    # Retraction: same shape but with adhesion offset + viscoelastic loss
    loss_factor = 1 - visc_loss_pct / 100
    F_ret = a * loss_factor * delta**1.5 - F_adh_nN * (1 - delta / max_indent_nm)
    F_ret = np.maximum(F_ret, -F_adh_nN)  # cap at adhesion force

    # Compute W_diss using trapezoidal rule
    dF = F_app - F_ret  # force difference at each point
    W_diss = trapezoid(dF, delta)  # nN·nm

    # Convert to SI
    W_diss_J = W_diss * 1e-18  # nN·nm → J

    # Conservative reference
    F_cons_ret = a * delta**1.5  # no loss
    W_cons = trapezoid(F_app - F_cons_ret, delta)

    print(f"  Hysteresis & Energy Dissipation")
    print(f"  ─────────────────────────────────")
    print(f"  Adhesion force         F_adh = {F_adh_nN:.2f} nN")
    print(f"  Viscoelastic loss            = {visc_loss_pct:.0f}%")
    print(f"  Max indentation              = {max_indent_nm:.0f} nm")
    print()
    print(f"  {'Quantity':<30s}  {'Value':>15s}")
    print(f"  {'─'*30}  {'─'*15}")
    print(f"  {'Dissipated energy W_diss':<30s}  {W_diss:>12.1f} nN·nm")
    print(f"  {'W_diss in SI':<30s}  {W_diss_J:>12.2e} J")
    print(f"  {'W_diss in kT (T=300K)':<30s}  {W_diss_J / (k_B * 300):>12.1f} kT")
    print(f"  {'Conservative reference':<30s}  {W_cons:>12.1f} nN·nm")

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # F-d with hysteresis
    axes[0].plot(delta, F_app, 'b-', lw=2.5, label='Approach')
    axes[0].plot(delta, F_ret, 'r--', lw=2.5, label='Retraction')
    axes[0].fill_between(delta, F_app, F_ret, alpha=0.2, color='green',
                         label=f'W_diss = {W_diss:.1f} nN·nm')
    axes[0].axhline(0, color='gray', ls=':', lw=0.8)
    axes[0].set_xlabel('Indentation δ (nm)')
    axes[0].set_ylabel('Force F (nN)')
    axes[0].set_title('Force–indentation with hysteresis')
    axes[0].legend(fontsize=8)

    # Conservative comparison
    axes[1].plot(delta, F_app, 'b-', lw=2.5, label='Approach')
    axes[1].plot(delta, F_cons_ret, 'b--', lw=2, alpha=0.5, label='Conservative retraction')
    axes[1].plot(delta, F_ret, 'r--', lw=2.5, label='Dissipative retraction')
    axes[1].set_xlabel('Indentation δ (nm)')
    axes[1].set_ylabel('Force F (nN)')
    axes[1].set_title('Conservative vs dissipative')
    axes[1].legend(fontsize=8)

    # W_diss vs adhesion sweep
    adh_range = np.linspace(0.1, 10, 50)
    W_sweep = []
    for fa in adh_range:
        F_r = a * loss_factor * delta**1.5 - fa * (1 - delta / max_indent_nm)
        F_r = np.maximum(F_r, -fa)
        W_sweep.append(trapezoid(F_app - F_r, delta))
    axes[2].plot(adh_range, W_sweep, 'g-', lw=2.5)
    axes[2].axvline(F_adh_nN, color='red', ls='--', lw=1, label=f'Current F_adh = {F_adh_nN:.1f} nN')
    axes[2].set_xlabel('Adhesion force (nN)')
    axes[2].set_ylabel('W_diss (nN·nm)')
    axes[2].set_title('Dissipated energy vs adhesion')
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

interact(
    interactive_hysteresis,
    F_adh_nN=FloatSlider(value=2.0, min=0.1, max=10, step=0.1, description='F_adh (nN)'),
    visc_loss_pct=FloatSlider(value=15, min=0, max=50, step=1, description='Loss (%)'),
    max_indent_nm=FloatSlider(value=30, min=5, max=100, step=5, description='δ_max (nm)')
);

---
## 4. Quantitative Parameter Extraction from Force Curves

A force–distance curve encodes multiple **quantitative parameters** (Section 5.4):

| Parameter | Symbol | Physical meaning |
|-----------|--------|------------------|
| Adhesion force | $F_{\mathrm{adh}}$ | Maximum attractive force during pull-off |
| Work of adhesion | $W_{\mathrm{adh}}$ | Energy to separate tip from surface |
| Contact stiffness | $k_{\mathrm{eff}}$ | Slope of the contact region $\mathrm{d}F / \mathrm{d}\delta$ |
| Energy dissipation | $W_{\mathrm{diss}}$ | Area of hysteresis loop |
| Contact point | CP | Transition from non-contact to contact |
| Indentation | $\Delta$ | Depth of tip penetration into sample |

**Key insight:** Each of these parameters carries different physical information. Extracting them correctly requires careful baseline subtraction, contact-point identification, and geometry correction.

### Tasks
1. Given a simulated force curve, extract all six parameters automatically.
2. Vary the material properties and see how each parameter responds.
3. Build intuition for which parameters are most sensitive to which properties.

In [ ]:
def interactive_parameter_extraction(E_kPa=30, F_adh_nN=3.0, visc_pct=10, R_nm=25):
    '''
    Extract quantitative parameters from a simulated force–indentation curve.
    '''
    R = R_nm * 1e-9
    E = E_kPa * 1e3  # Pa

    delta = np.linspace(0, 50, 600)  # nm
    delta_m = delta * 1e-9

    # Hertz approach
    F_app = (4/3) * E * np.sqrt(R) * delta_m**1.5 * 1e9  # nN

    # Retraction with adhesion + viscoelastic loss
    loss = 1 - visc_pct / 100
    F_ret = (4/3) * E * loss * np.sqrt(R) * delta_m**1.5 * 1e9 - F_adh_nN * np.exp(-delta / 15)

    # ── Extract parameters ──
    # 1. Adhesion force
    F_adh_extracted = -np.min(F_ret)

    # 2. Work of adhesion (area below F=0 on retraction)
    neg_mask = F_ret < 0
    if np.any(neg_mask):
        W_adh = -trapezoid(F_ret[neg_mask], delta[neg_mask])  # nN·nm
    else:
        W_adh = 0

    # 3. Contact stiffness (slope at δ = 20 nm)
    idx_20 = np.argmin(np.abs(delta - 20))
    k_eff = np.gradient(F_app, delta)[idx_20]  # nN/nm

    # 4. Energy dissipation
    W_diss = trapezoid(F_app - F_ret, delta)

    # 5. Max indentation at max force
    delta_max = delta[-1]

    print(f"  Quantitative Parameter Extraction")
    print(f"  ─────────────────────────────────")
    print(f"  Material: E* = {E_kPa:.0f} kPa, R = {R_nm:.0f} nm")
    print()
    print(f"  {'Parameter':<28s}  {'Value':>12s}  {'Unit':>10s}")
    print(f"  {'─'*28}  {'─'*12}  {'─'*10}")
    print(f"  {'Adhesion force F_adh':<28s}  {F_adh_extracted:>12.2f}  {'nN':>10s}")
    print(f"  {'Work of adhesion W_adh':<28s}  {W_adh:>12.1f}  {'nN·nm':>10s}")
    print(f"  {'Contact stiffness k_eff':<28s}  {k_eff:>12.3f}  {'nN/nm':>10s}")
    print(f"  {'Energy dissipation W_diss':<28s}  {W_diss:>12.1f}  {'nN·nm':>10s}")
    print(f"  {'Max indentation Δ':<28s}  {delta_max:>12.0f}  {'nm':>10s}")

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    # Full curve with all annotations
    ax = axes[0, 0]
    ax.plot(delta, F_app, 'b-', lw=2, label='Approach')
    ax.plot(delta, F_ret, 'r--', lw=2, label='Retraction')
    ax.fill_between(delta, F_app, F_ret, alpha=0.1, color='purple', label='W_diss')
    if np.any(neg_mask):
        ax.fill_between(delta[neg_mask], F_ret[neg_mask], 0, alpha=0.2, color='red', label='W_adh')
    ax.axhline(0, color='gray', ls=':', lw=0.8)
    # Adhesion arrow
    min_idx = np.argmin(F_ret)
    ax.annotate('', xy=(delta[min_idx], F_ret[min_idx]), xytext=(delta[min_idx], 0),
                arrowprops=dict(arrowstyle='<->', color='red', lw=1.5))
    ax.text(delta[min_idx]+2, F_ret[min_idx]/2, f'F_adh = {F_adh_extracted:.2f} nN',
            fontsize=8, color='red')
    ax.set_xlabel('Indentation δ (nm)')
    ax.set_ylabel('Force (nN)')
    ax.set_title('Annotated force–indentation curve')
    ax.legend(fontsize=7)

    # Stiffness (slope)
    ax = axes[0, 1]
    grad = np.gradient(F_app, delta)
    ax.plot(delta, grad, 'b-', lw=2)
    ax.axvline(20, color='gray', ls='--', lw=1)
    ax.plot(20, k_eff, 'ro', ms=8, label=f'k_eff(δ=20nm) = {k_eff:.3f} nN/nm')
    ax.set_xlabel('Indentation δ (nm)')
    ax.set_ylabel('dF/dδ (nN/nm)')
    ax.set_title('Contact stiffness vs indentation')
    ax.legend(fontsize=8)

    # Parameter sensitivity: E sweep
    ax = axes[1, 0]
    E_range = np.linspace(5, 200, 40)
    F_adh_list, k_list, W_list = [], [], []
    for Ev in E_range:
        Fm = (4/3) * Ev * 1e3 * np.sqrt(R) * delta_m**1.5 * 1e9
        k_list.append(np.gradient(Fm, delta)[idx_20])
    ax.plot(E_range, k_list, 'b-', lw=2)
    ax.axvline(E_kPa, color='red', ls='--', lw=1, label=f'Current E* = {E_kPa} kPa')
    ax.set_xlabel('Young\'s modulus E* (kPa)')
    ax.set_ylabel('k_eff at δ=20nm (nN/nm)')
    ax.set_title('Stiffness sensitivity to E*')
    ax.legend(fontsize=8)

    # Energy balance
    ax = axes[1, 1]
    W_elastic = trapezoid(F_app, delta)
    W_recovered = trapezoid(np.maximum(F_ret, 0), delta)
    bars = ax.bar(['Elastic\n(stored)', 'Recovered', 'Dissipated', 'Adhesion'],
                  [W_elastic, W_recovered, W_diss, W_adh],
                  color=['#2F6DB3', '#4C9A7D', '#9B59B6', '#C84A4A'])
    ax.set_ylabel('Energy (nN·nm)')
    ax.set_title('Energy balance')

    plt.tight_layout()
    plt.show()

interact(
    interactive_parameter_extraction,
    E_kPa=FloatSlider(value=30, min=1, max=200, step=5, description='E* (kPa)'),
    F_adh_nN=FloatSlider(value=3.0, min=0, max=10, step=0.2, description='F_adh (nN)'),
    visc_pct=FloatSlider(value=10, min=0, max=40, step=1, description='Loss (%)'),
    R_nm=FloatSlider(value=25, min=5, max=100, step=5, description='R_tip (nm)')
);

---
## 5. Contact Point Detection: Why It Matters

The **contact point** is the transition between non-contact and contact regimes. Its accurate identification is critical because all subsequent analysis (indentation, modulus extraction) depends on it (Section 5.6.5).

Common detection methods:
- **Threshold method**: first point where $F > F_{\mathrm{threshold}}$
- **Ratio of variance (RoV)**: detect change in signal variance
- **Fitting method**: fit baseline + contact model simultaneously

The Hertz model predicts $F \propto \delta^{3/2}$, so a contact-point error $\epsilon$ propagates as:

$$E^* \propto \delta^{-3/2} \quad \Rightarrow \quad \text{small } \epsilon \text{ in } \delta \;\to\; \text{large error in } E^*$$

**Key insight:** Contact-point uncertainty is the **single largest source of error** in modulus extraction (Section 5.6, Conceptual Insight).

### Tasks
1. Implement threshold and RoV contact-point detection.
2. Shift the contact point artificially and observe the effect on $F(\delta)$.
3. Quantify the modulus error as a function of contact-point shift.

In [ ]:
def interactive_contact_point(noise_nm=0.3, threshold_nN=0.05, cp_shift_nm=0):
    '''
    Contact point detection and its effect on modulus extraction.
    '''
    # Generate a realistic force curve with noise
    z_piezo = np.linspace(100, -40, 800)  # nm
    E_true = 20e3  # Pa (20 kPa)
    R = 25e-9  # m

    # True contact at z=0
    d_true = np.zeros_like(z_piezo)
    contact = z_piezo < 0
    indent_true = np.abs(z_piezo[contact]) * 1e-9
    d_true[contact] = (4/3) * E_true * np.sqrt(R) * indent_true**1.5 / 0.3 * 1e9 / (0.3 * 1e9)

    # Force = Hertz in contact, zero baseline elsewhere
    F_true = np.zeros_like(z_piezo)
    F_true[contact] = (4/3) * E_true * np.sqrt(R) * indent_true**1.5 * 1e9

    # Add noise
    rng = np.random.default_rng(42)
    F_noisy = F_true + noise_nm * rng.normal(size=len(z_piezo))

    # ── Contact point detection: threshold ──
    above_thresh = np.where(F_noisy > threshold_nN)[0]
    if len(above_thresh) > 0:
        cp_thresh_idx = above_thresh[0]
    else:
        cp_thresh_idx = len(z_piezo) // 2
    cp_thresh = z_piezo[cp_thresh_idx]

    # ── Contact point detection: ratio of variance ──
    window = 20
    rov = np.zeros(len(F_noisy))
    for i in range(window, len(F_noisy) - window):
        var_before = np.var(F_noisy[i-window:i])
        var_after = np.var(F_noisy[i:i+window])
        rov[i] = var_after / (var_before + 1e-12)
    cp_rov_idx = np.argmax(rov)
    cp_rov = z_piezo[cp_rov_idx]

    # ── Effect of contact point shift on modulus ──
    # True indentation
    cp_true = 0  # nm
    cp_shifted = cp_true + cp_shift_nm

    delta_true = np.maximum(cp_true - z_piezo, 0)
    delta_shifted = np.maximum(cp_shifted - z_piezo, 0)

    # Hertz fit on the shifted data
    fit_mask_true = delta_true > 2
    fit_mask_shift = delta_shifted > 2

    def hertz(delta_nm, E_Pa):
        return (4/3) * E_Pa * np.sqrt(R) * (delta_nm * 1e-9)**1.5 * 1e9

    try:
        popt_true, _ = curve_fit(hertz, delta_true[fit_mask_true], F_noisy[fit_mask_true], p0=[20e3])
        E_fit_true = popt_true[0]
    except:
        E_fit_true = E_true

    try:
        popt_shift, _ = curve_fit(hertz, delta_shifted[fit_mask_shift], F_noisy[fit_mask_shift], p0=[20e3])
        E_fit_shift = popt_shift[0]
    except:
        E_fit_shift = E_true

    E_error_pct = (E_fit_shift / E_true - 1) * 100

    print(f"  Contact Point Detection & Error Propagation")
    print(f"  ─────────────────────────────────────────────")
    print(f"  True contact point    = 0 nm")
    print(f"  Threshold detection   = {cp_thresh:.1f} nm  (F > {threshold_nN:.2f} nN)")
    print(f"  RoV detection         = {cp_rov:.1f} nm")
    print(f"  Manual shift applied  = {cp_shift_nm:+.1f} nm")
    print()
    print(f"  {'Modulus (true CP)':<30s}  E* = {E_fit_true/1e3:.1f} kPa")
    print(f"  {'Modulus (shifted CP)':<30s}  E* = {E_fit_shift/1e3:.1f} kPa")
    print(f"  {'Error from CP shift':<30s}  {E_error_pct:+.1f}%")

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    # Force curve with detected CPs
    ax = axes[0, 0]
    ax.plot(z_piezo, F_noisy, 'k-', lw=1, alpha=0.6, label='Noisy data')
    ax.plot(z_piezo, F_true, 'b-', lw=1.5, alpha=0.4, label='True curve')
    ax.axvline(0, color='green', ls='-', lw=2, label='True CP')
    ax.axvline(cp_thresh, color='red', ls='--', lw=1.5, label=f'Threshold CP = {cp_thresh:.1f} nm')
    ax.axvline(cp_rov, color='orange', ls='-.', lw=1.5, label=f'RoV CP = {cp_rov:.1f} nm')
    ax.set_xlabel('z-piezo (nm)')
    ax.set_ylabel('Force (nN)')
    ax.set_title('Contact point detection')
    ax.legend(fontsize=7)
    ax.set_xlim(-40, 40)

    # RoV signal
    ax = axes[0, 1]
    ax.plot(z_piezo, rov, 'orange', lw=2)
    ax.axvline(cp_rov, color='orange', ls='--', lw=1.5)
    ax.set_xlabel('z-piezo (nm)')
    ax.set_ylabel('Ratio of Variance')
    ax.set_title('RoV contact point detection')
    ax.set_xlim(-40, 40)

    # F(δ) with true vs shifted CP
    ax = axes[1, 0]
    ax.plot(delta_true, F_noisy, 'b-', lw=1.5, alpha=0.6, label='True CP')
    ax.plot(delta_shifted, F_noisy, 'r-', lw=1.5, alpha=0.6, label=f'Shifted CP ({cp_shift_nm:+.0f} nm)')
    delta_fit = np.linspace(0, 40, 200)
    ax.plot(delta_fit, hertz(delta_fit, E_fit_true), 'b--', lw=1, label=f'Hertz fit: {E_fit_true/1e3:.1f} kPa')
    ax.plot(delta_fit, hertz(delta_fit, E_fit_shift), 'r--', lw=1, label=f'Hertz fit: {E_fit_shift/1e3:.1f} kPa')
    ax.set_xlabel('Indentation δ (nm)')
    ax.set_ylabel('Force (nN)')
    ax.set_title('Effect of CP shift on F(δ)')
    ax.legend(fontsize=7)

    # E* error vs CP shift sweep
    ax = axes[1, 1]
    shifts = np.linspace(-5, 5, 41)
    E_errors = []
    for s in shifts:
        d_s = np.maximum(s - z_piezo, 0)
        m = d_s > 2
        if np.sum(m) > 5:
            try:
                p, _ = curve_fit(hertz, d_s[m], F_noisy[m], p0=[20e3])
                E_errors.append((p[0] / E_true - 1) * 100)
            except:
                E_errors.append(np.nan)
        else:
            E_errors.append(np.nan)
    ax.plot(shifts, E_errors, 'b-', lw=2.5)
    ax.axhline(0, color='gray', ls=':', lw=0.8)
    ax.axvline(0, color='green', ls='-', lw=1.5, alpha=0.5)
    ax.axvline(cp_shift_nm, color='red', ls='--', lw=1.5)
    ax.set_xlabel('Contact point shift (nm)')
    ax.set_ylabel('E* error (%)')
    ax.set_title('Modulus error vs contact point uncertainty')

    plt.tight_layout()
    plt.show()

interact(
    interactive_contact_point,
    noise_nm=FloatSlider(value=0.3, min=0, max=2, step=0.05, description='Noise (nN)'),
    threshold_nN=FloatSlider(value=0.05, min=0.01, max=1.0, step=0.01, description='Threshold (nN)'),
    cp_shift_nm=FloatSlider(value=0, min=-5, max=5, step=0.2, description='CP shift (nm)')
);

---
## 6. Hertz Model Fitting: From $F(\delta)$ to Young's Modulus

The **Hertz model** describes elastic contact between a sphere and a flat surface (Section 5.8):

$$F = \frac{4}{3} E^* \sqrt{R} \;\delta^{3/2}$$

where $E^*$ is the reduced Young's modulus, $R$ the tip radius, and $\delta$ the indentation depth.

For a **conical tip** (half-angle $\alpha$):

$$F = \frac{2}{\pi} E^* \tan(\alpha) \;\delta^2$$

By fitting the measured $F(\delta)$ curve, we extract $E^*$. The quality of the fit depends on noise, indentation range, and tip geometry.

**Key insight:** The Hertz model assumes purely elastic, adhesion-free contact. For soft biological samples with significant adhesion, JKR or DMT models (Chapter 6) may be more appropriate.

### Tasks
1. Generate noisy Hertz data with known $E^*$ and fit it back.
2. Compare spherical vs conical tip geometry.
3. Explore how noise level and fitting range affect the extracted modulus.

In [ ]:
def interactive_hertz_fit(E_true_kPa=20, R_nm=25, noise_pct=5, fit_max_nm=30, tip_shape='Sphere'):
    '''
    Fit Hertz model to synthetic force-indentation data.
    '''
    R = R_nm * 1e-9
    E_true = E_true_kPa * 1e3  # Pa
    alpha = np.radians(18)  # cone half-angle

    delta = np.linspace(0.5, 50, 300)  # nm
    delta_m = delta * 1e-9

    # True force
    if tip_shape == 'Sphere':
        F_true = (4/3) * E_true * np.sqrt(R) * delta_m**1.5 * 1e9
    else:  # Cone
        F_true = (2/np.pi) * E_true * np.tan(alpha) * delta_m**2 * 1e9

    # Add noise
    rng = np.random.default_rng(7)
    noise_level = noise_pct / 100 * np.max(F_true)
    F_noisy = F_true + noise_level * rng.normal(size=len(delta))

    # Fit functions
    def hertz_sphere(d_nm, E_Pa):
        return (4/3) * E_Pa * np.sqrt(R) * (d_nm * 1e-9)**1.5 * 1e9

    def hertz_cone(d_nm, E_Pa):
        return (2/np.pi) * E_Pa * np.tan(alpha) * (d_nm * 1e-9)**2 * 1e9

    # Select fitting range
    fit_mask = delta <= fit_max_nm
    fit_func = hertz_sphere if tip_shape == 'Sphere' else hertz_cone

    try:
        popt, pcov = curve_fit(fit_func, delta[fit_mask], F_noisy[fit_mask], p0=[E_true])
        E_fit = popt[0]
        E_std = np.sqrt(pcov[0, 0])
    except:
        E_fit = E_true
        E_std = 0

    # Residuals
    F_fit = fit_func(delta, E_fit)
    residuals = F_noisy - F_fit
    rmse = np.sqrt(np.mean(residuals[fit_mask]**2))

    print(f"  Hertz Model Fitting ({tip_shape})")
    print(f"  ─────────────────────────────────")
    print(f"  True modulus      E* = {E_true_kPa:.1f} kPa")
    print(f"  Fitted modulus    E* = {E_fit/1e3:.2f} ± {E_std/1e3:.2f} kPa")
    print(f"  Error                = {(E_fit/E_true - 1)*100:+.1f}%")
    print(f"  Fit range            = 0 – {fit_max_nm:.0f} nm")
    print(f"  RMSE                 = {rmse:.3f} nN")
    print(f"  Noise level          = {noise_pct:.0f}% of max force")

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Data + fit
    axes[0].scatter(delta, F_noisy, s=8, color='gray', alpha=0.5, label='Noisy data')
    axes[0].plot(delta, F_true, 'k-', lw=1, alpha=0.3, label=f'True (E*={E_true_kPa} kPa)')
    axes[0].plot(delta, F_fit, 'b-', lw=2.5, label=f'Fit (E*={E_fit/1e3:.1f} kPa)')
    axes[0].axvline(fit_max_nm, color='red', ls='--', lw=1, label=f'Fit range limit')
    axes[0].set_xlabel('Indentation δ (nm)')
    axes[0].set_ylabel('Force (nN)')
    axes[0].set_title('Force–indentation fit')
    axes[0].legend(fontsize=8)

    # Residuals
    axes[1].scatter(delta[fit_mask], residuals[fit_mask], s=10, color='blue', alpha=0.5)
    axes[1].axhline(0, color='gray', ls=':', lw=1)
    axes[1].set_xlabel('Indentation δ (nm)')
    axes[1].set_ylabel('Residual (nN)')
    axes[1].set_title(f'Residuals (RMSE = {rmse:.3f} nN)')

    # E* vs fit range
    ranges = np.arange(5, 50, 2)
    E_fits = []
    for r in ranges:
        m = delta <= r
        if np.sum(m) > 3:
            try:
                p, _ = curve_fit(fit_func, delta[m], F_noisy[m], p0=[E_true])
                E_fits.append(p[0] / 1e3)
            except:
                E_fits.append(np.nan)
        else:
            E_fits.append(np.nan)
    axes[2].plot(ranges, E_fits, 'b-o', lw=2, ms=4)
    axes[2].axhline(E_true_kPa, color='green', ls='--', lw=1.5, label=f'True E* = {E_true_kPa} kPa')
    axes[2].axvline(fit_max_nm, color='red', ls='--', lw=1)
    axes[2].set_xlabel('Fit range limit (nm)')
    axes[2].set_ylabel('Fitted E* (kPa)')
    axes[2].set_title('Modulus stability vs fit range')
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

interact(
    interactive_hertz_fit,
    E_true_kPa=FloatSlider(value=20, min=1, max=200, step=1, description='E* true (kPa)'),
    R_nm=FloatSlider(value=25, min=5, max=100, step=5, description='R_tip (nm)'),
    noise_pct=FloatSlider(value=5, min=0, max=30, step=1, description='Noise (%)'),
    fit_max_nm=FloatSlider(value=30, min=5, max=50, step=1, description='Fit range (nm)'),
    tip_shape=Dropdown(options=['Sphere', 'Cone'], value='Sphere', description='Tip shape')
);

---
## 7. Contact Mechanics Models: Hertz vs JKR vs DMT

The same force–indentation data can be interpreted with different contact mechanics models (Section 5.8):

| Model | Adhesion | Best for | Key equation |
|-------|----------|----------|--------------|
| **Hertz** | None | Stiff samples, low adhesion | $F = \frac{4}{3}E^*\sqrt{R}\,\delta^{3/2}$ |
| **JKR** | Strong (short-range) | Soft, compliant samples | Includes surface energy $\gamma$ |
| **DMT** | Weak (long-range) | Stiff samples, weak adhesion | $F = \frac{4}{3}E^*\sqrt{R}\,\delta^{3/2} - 2\pi R \gamma$ |

The **Tabor parameter** $\mu_T$ guides model selection:

$$\mu_T = \left(\frac{R \gamma^2}{E^{*2} z_0^3}\right)^{1/3}$$

- $\mu_T \gg 1$: JKR regime (soft, adhesive)
- $\mu_T \ll 1$: DMT regime (stiff, weakly adhesive)

**Key insight:** The choice of contact model is not optional — it directly affects the extracted Young's modulus (Figure 5.8).

### Tasks
1. Compare Hertz, JKR-approximation, and DMT fits to the same data.
2. Observe how extracted modulus depends on model choice.
3. Explore the Tabor parameter space to understand model selection.

In [ ]:
def interactive_contact_models(E_kPa=20, gamma_mJ_m2=30, R_nm=25, z0_nm=0.3):
    '''
    Compare Hertz, JKR, and DMT contact models on the same data.
    '''
    R = R_nm * 1e-9
    E_star = E_kPa * 1e3
    gamma = gamma_mJ_m2 * 1e-3  # J/m^2
    z0 = z0_nm * 1e-9

    # Tabor parameter
    mu_T = (R * gamma**2 / (E_star**2 * z0**3))**(1/3)

    # Generate "true" data using JKR-like behaviour
    delta = np.linspace(0, 40, 400)  # nm
    delta_m = delta * 1e-9

    # Hertz component
    F_hertz = (4/3) * E_star * np.sqrt(R) * delta_m**1.5 * 1e9  # nN

    # JKR approximation: F = Hertz - adhesion pull
    F_adh_JKR = 1.5 * np.pi * R * gamma * 1e9  # nN (JKR: 3/2 * pi * R * gamma)
    F_jkr = (4/3) * E_star * 0.85 * np.sqrt(R) * delta_m**1.5 * 1e9 - F_adh_JKR * np.exp(-delta / 5)

    # DMT: Hertz + constant adhesion offset
    F_adh_DMT = 2 * np.pi * R * gamma * 1e9  # nN (DMT: 2*pi*R*gamma)
    F_dmt = (4/3) * E_star * np.sqrt(R) * delta_m**1.5 * 1e9 - F_adh_DMT

    # "Experimental" data — mixture with noise
    # Weight between JKR and DMT based on Tabor parameter
    w_jkr = np.clip(mu_T / 5, 0, 1)  # more JKR-like for high mu_T
    F_data = w_jkr * F_jkr + (1 - w_jkr) * F_dmt
    rng = np.random.default_rng(42)
    F_data += 0.02 * np.max(np.abs(F_data)) * rng.normal(size=len(delta))

    # Fit each model to the data
    def hertz_model(d, E): return (4/3) * E * np.sqrt(R) * (d*1e-9)**1.5 * 1e9
    def dmt_model(d, E): return (4/3) * E * np.sqrt(R) * (d*1e-9)**1.5 * 1e9 - F_adh_DMT

    fit_m = delta > 3
    try:
        E_h, = curve_fit(hertz_model, delta[fit_m], F_data[fit_m], p0=[E_star])[0]
    except: E_h = E_star
    try:
        E_d, = curve_fit(dmt_model, delta[fit_m], F_data[fit_m], p0=[E_star])[0]
    except: E_d = E_star

    # Model selection guidance
    if mu_T > 3:
        regime = "JKR (soft, adhesive)"
    elif mu_T < 0.1:
        regime = "DMT (stiff, low adhesion)"
    else:
        regime = "Transition (Maugis)"

    print(f"  Contact Mechanics Model Comparison")
    print(f"  ─────────────────────────────────")
    print(f"  E* = {E_kPa:.0f} kPa,  γ = {gamma_mJ_m2:.0f} mJ/m²,  R = {R_nm:.0f} nm")
    print(f"  Tabor parameter  μ_T = {mu_T:.2f}  →  {regime}")
    print()
    print(f"  {'Model':<12s}  {'Fitted E* (kPa)':>16s}  {'F_adh (nN)':>12s}")
    print(f"  {'─'*12}  {'─'*16}  {'─'*12}")
    print(f"  {'Hertz':<12s}  {E_h/1e3:>16.1f}  {'—':>12s}")
    print(f"  {'DMT':<12s}  {E_d/1e3:>16.1f}  {F_adh_DMT:>12.2f}")
    print(f"  {'JKR':<12s}  {'—':>16s}  {F_adh_JKR:>12.2f}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Model comparison
    axes[0].scatter(delta, F_data, s=6, color='gray', alpha=0.4, label='Data')
    axes[0].plot(delta, hertz_model(delta, E_h), 'b-', lw=2, label=f'Hertz ({E_h/1e3:.1f} kPa)')
    axes[0].plot(delta, F_jkr, 'g-', lw=2, label='JKR-like')
    axes[0].plot(delta, dmt_model(delta, E_d), 'r-', lw=2, label=f'DMT ({E_d/1e3:.1f} kPa)')
    axes[0].axhline(0, color='gray', ls=':', lw=0.8)
    axes[0].set_xlabel('Indentation δ (nm)')
    axes[0].set_ylabel('Force (nN)')
    axes[0].set_title('Three models, same data')
    axes[0].legend(fontsize=8)

    # Bar comparison of extracted E*
    models = ['Hertz', 'DMT', 'True']
    E_vals = [E_h/1e3, E_d/1e3, E_kPa]
    colors = ['#2F6DB3', '#C84A4A', '#333333']
    axes[1].bar(models, E_vals, color=colors, edgecolor='white', width=0.5)
    for i, v in enumerate(E_vals):
        axes[1].text(i, v + 0.5, f'{v:.1f}', ha='center', fontsize=9)
    axes[1].set_ylabel("Fitted E* (kPa)")
    axes[1].set_title('Model choice → different E*')

    # Tabor parameter map
    E_range = np.logspace(2, 6, 100)  # Pa
    gamma_range = np.logspace(-3, 0, 100)  # J/m^2
    E_grid, G_grid = np.meshgrid(E_range, gamma_range)
    mu_grid = (R * G_grid**2 / (E_grid**2 * z0**3))**(1/3)
    c = axes[2].pcolormesh(E_grid/1e3, G_grid*1e3, np.log10(mu_grid),
                           cmap='RdYlBu_r', shading='auto')
    axes[2].set_xscale('log')
    axes[2].set_yscale('log')
    axes[2].plot(E_kPa, gamma_mJ_m2, 'k*', ms=15, zorder=5, label=f'μ_T = {mu_T:.2f}')
    axes[2].set_xlabel('E* (kPa)')
    axes[2].set_ylabel('γ (mJ/m²)')
    axes[2].set_title('Tabor parameter map')
    fig.colorbar(c, ax=axes[2], label='log₁₀(μ_T)')
    axes[2].legend(fontsize=9)

    plt.tight_layout()
    plt.show()

interact(
    interactive_contact_models,
    E_kPa=FloatLogSlider(value=20, base=10, min=0, max=3, step=0.1, description='E* (kPa)'),
    gamma_mJ_m2=FloatSlider(value=30, min=1, max=200, step=5, description='γ (mJ/m²)'),
    R_nm=FloatSlider(value=25, min=5, max=100, step=5, description='R_tip (nm)'),
    z0_nm=FloatSlider(value=0.3, min=0.1, max=1.0, step=0.05, description='z₀ (nm)')
);

---
## 8. Force Mapping: From Single Curve to Spatial Maps

In **force volume imaging** (Section 5.7), a force–distance curve is acquired at every pixel of a grid. From each curve, quantitative parameters are extracted and displayed as 2D maps:

- **Topography** from contact point position
- **Adhesion map** from pull-off force
- **Modulus map** from Hertz fitting
- **Dissipation map** from hysteresis area

This extends single-point spectroscopy to **spatially resolved nanomechanical mapping**.

**Key insight:** Force mapping turns a 1D measurement into a 3D dataset $F(x, y, \delta)$, enabling spatial correlation between topography and mechanical properties.

### Tasks
1. Generate a grid of force curves on a heterogeneous sample.
2. Extract adhesion and modulus at each pixel.
3. Construct and display 2D parameter maps.

In [ ]:
def interactive_force_mapping(grid_size=16, E_soft_kPa=5, E_hard_kPa=100, adh_soft_nN=5, adh_hard_nN=1):
    '''
    Simulate force volume imaging on a two-region sample.
    '''
    N = grid_size
    R = 25e-9  # tip radius

    # Create a sample with two regions: soft (circle) and hard (background)
    x = np.arange(N)
    y = np.arange(N)
    X, Y = np.meshgrid(x, y)
    cx, cy = N / 2, N / 2
    dist = np.sqrt((X - cx)**2 + (Y - cy)**2)
    is_soft = dist < N / 3

    # Maps to store results
    E_map = np.zeros((N, N))
    adh_map = np.zeros((N, N))
    topo_map = np.zeros((N, N))

    rng = np.random.default_rng(42)

    for i in range(N):
        for j in range(N):
            if is_soft[i, j]:
                E_local = E_soft_kPa * 1e3 * (1 + 0.1 * rng.normal())
                F_adh = adh_soft_nN * (1 + 0.1 * rng.normal())
                topo_offset = -3 + 0.5 * rng.normal()  # soft region is lower
            else:
                E_local = E_hard_kPa * 1e3 * (1 + 0.05 * rng.normal())
                F_adh = adh_hard_nN * (1 + 0.1 * rng.normal())
                topo_offset = 0.5 * rng.normal()

            E_map[i, j] = E_local / 1e3  # kPa
            adh_map[i, j] = F_adh
            topo_map[i, j] = topo_offset

    print(f"  Force Volume Imaging Simulation")
    print(f"  ─────────────────────────────────")
    print(f"  Grid size         = {N} × {N} ({N*N} curves)")
    print(f"  Soft region:  E* = {E_soft_kPa} kPa,  F_adh = {adh_soft_nN} nN")
    print(f"  Hard region:  E* = {E_hard_kPa} kPa,  F_adh = {adh_hard_nN} nN")
    print()
    print(f"  {'Map':<15s}  {'Soft (mean)':>12s}  {'Hard (mean)':>12s}  {'Contrast':>10s}")
    print(f"  {'─'*15}  {'─'*12}  {'─'*12}  {'─'*10}")
    E_soft_mean = np.mean(E_map[is_soft])
    E_hard_mean = np.mean(E_map[~is_soft])
    adh_soft_mean = np.mean(adh_map[is_soft])
    adh_hard_mean = np.mean(adh_map[~is_soft])
    print(f"  {'Modulus (kPa)':<15s}  {E_soft_mean:>12.1f}  {E_hard_mean:>12.1f}  {E_hard_mean/E_soft_mean:>10.1f}×")
    print(f"  {'Adhesion (nN)':<15s}  {adh_soft_mean:>12.2f}  {adh_hard_mean:>12.2f}  {adh_soft_mean/adh_hard_mean:>10.1f}×")

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    # Sample structure
    axes[0].imshow(is_soft.astype(float), cmap='coolwarm', origin='lower', vmin=0, vmax=1)
    axes[0].set_title('Sample (blue=soft)')
    axes[0].set_xlabel('x pixel')
    axes[0].set_ylabel('y pixel')

    # Topography
    im1 = axes[1].imshow(topo_map, cmap='gray', origin='lower')
    axes[1].set_title('Topography (nm)')
    fig.colorbar(im1, ax=axes[1], shrink=0.8)

    # Adhesion map
    im2 = axes[2].imshow(adh_map, cmap='Reds', origin='lower')
    axes[2].set_title('Adhesion (nN)')
    fig.colorbar(im2, ax=axes[2], shrink=0.8)

    # Modulus map
    im3 = axes[3].imshow(E_map, cmap='YlGnBu', origin='lower')
    axes[3].set_title('Modulus E* (kPa)')
    fig.colorbar(im3, ax=axes[3], shrink=0.8)

    for ax in axes:
        ax.set_xlabel('x pixel')

    plt.tight_layout()
    plt.show()

interact(
    interactive_force_mapping,
    grid_size=IntSlider(value=16, min=8, max=32, step=4, description='Grid size'),
    E_soft_kPa=FloatSlider(value=5, min=1, max=50, step=1, description='E* soft (kPa)'),
    E_hard_kPa=FloatSlider(value=100, min=20, max=500, step=10, description='E* hard (kPa)'),
    adh_soft_nN=FloatSlider(value=5, min=0.5, max=20, step=0.5, description='F_adh soft (nN)'),
    adh_hard_nN=FloatSlider(value=1, min=0.1, max=5, step=0.1, description='F_adh hard (nN)')
);

---
## 9. Measuring Bacterial Stiffness: An AFM Design Problem

Measuring the mechanical properties of bacteria requires careful experimental design (Section 5.9). Key constraints:

- **Cantilever stiffness** must be comparable to the cell ($k \sim 0.01$–$0.5$ N/m)
- **Applied force** must be low enough to avoid cell damage ($F < 1$ nN typical)
- **Indentation** must remain small relative to cell height ($\delta / h_{\mathrm{cell}} < 10\%$)
- **Measurements must be reproducible** across multiple curves

The cell's **apparent stiffness** depends on the measurement conditions (force, indentation depth, tip geometry), making this an engineering optimization problem.

**Key insight:** AFM nanomechanics on soft biological samples is not a single measurement — it is an iterative process of cantilever selection, force optimization, and reproducibility validation (Figure 5.9).

### Tasks
1. Simulate indentation of a bacterial cell with realistic parameters.
2. Check the 10% indentation rule for different force setpoints.
3. Explore how cantilever stiffness affects the measurement.

In [ ]:
def interactive_bacterial_stiffness(k_cant=0.06, F_max_nN=1.0, h_cell_nm=800, E_cell_kPa=20, R_nm=25):
    '''
    Simulate AFM measurement on a single bacterium — check design constraints.
    '''
    R = R_nm * 1e-9
    E = E_cell_kPa * 1e3
    k = k_cant

    # Max indentation from Hertz: F = (4/3)*E*sqrt(R)*delta^1.5
    # delta = (3*F / (4*E*sqrt(R)))^(2/3)
    F_max = F_max_nN * 1e-9  # N
    delta_max = (3 * F_max / (4 * E * np.sqrt(R)))**(2/3) * 1e9  # nm

    # Indentation ratio
    ratio = delta_max / h_cell_nm * 100  # percent

    # Cantilever deflection at max force
    d_cant = F_max_nN / k  # nm

    # Signal quality: is deflection measurable?
    signal_ok = d_cant > 1.0  # need > 1 nm deflection

    # Indentation rule
    indent_ok = ratio < 10

    # Force curve simulation
    delta = np.linspace(0, delta_max * 1.5, 300)
    delta_m = delta * 1e-9
    F_curve = (4/3) * E * np.sqrt(R) * delta_m**1.5 * 1e9  # nN

    # Deflection sensitivity: what fraction of z is deflection vs indentation?
    z_total = delta + F_curve / k  # total piezo displacement (nm)
    frac_indent = delta / (z_total + 1e-12) * 100
    frac_defl = 100 - frac_indent

    # Status
    status = "✓ VALID" if (signal_ok and indent_ok) else "✗ ADJUST"
    issues = []
    if not signal_ok: issues.append("Signal too weak (deflection < 1 nm)")
    if not indent_ok: issues.append(f"Indentation too deep ({ratio:.1f}% > 10%)")

    print(f"  Bacterial Stiffness Measurement Design")
    print(f"  ─────────────────────────────────────────")
    print(f"  Cell height           h    = {h_cell_nm:.0f} nm")
    print(f"  Cell modulus          E*   = {E_cell_kPa:.0f} kPa")
    print(f"  Cantilever stiffness  k    = {k_cant:.3f} N/m")
    print(f"  Max applied force     F    = {F_max_nN:.2f} nN")
    print(f"  Tip radius            R    = {R_nm:.0f} nm")
    print()
    print(f"  {'Result':<35s}  {'Value':>12s}  {'Status':>8s}")
    print(f"  {'─'*35}  {'─'*12}  {'─'*8}")
    print(f"  {'Max indentation δ':<35s}  {delta_max:>10.1f} nm  {'':>8s}")
    print(f"  {'Indentation ratio δ/h':<35s}  {ratio:>10.1f} %  {'OK' if indent_ok else 'HIGH':>8s}")
    print(f"  {'Cantilever deflection d':<35s}  {d_cant:>10.1f} nm  {'OK' if signal_ok else 'LOW':>8s}")
    print(f"  {'Measurement status':<35s}  {'':>12s}  {status:>8s}")
    for iss in issues:
        print(f"    ⚠  {iss}")

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    # Force-indentation curve with limits
    ax = axes[0, 0]
    ax.plot(delta, F_curve, 'b-', lw=2.5)
    ax.axhline(F_max_nN, color='red', ls='--', lw=1.5, label=f'F_max = {F_max_nN:.1f} nN')
    ax.axvline(h_cell_nm * 0.10, color='orange', ls='--', lw=1.5, label=f'10% rule = {h_cell_nm*0.1:.0f} nm')
    ax.axvline(delta_max, color='gray', ls=':', lw=1, label=f'δ_max = {delta_max:.1f} nm')
    ax.set_xlabel('Indentation δ (nm)')
    ax.set_ylabel('Force (nN)')
    ax.set_title('Force–indentation on bacterium')
    ax.legend(fontsize=8)

    # Partition: deflection vs indentation
    ax = axes[0, 1]
    ax.fill_between(z_total, 0, frac_indent, alpha=0.3, color='green', label='Indentation')
    ax.fill_between(z_total, frac_indent, 100, alpha=0.3, color='orange', label='Deflection')
    ax.set_xlabel('Total z-piezo motion (nm)')
    ax.set_ylabel('Fraction (%)')
    ax.set_title('z_piezo = δ + d partition')
    ax.legend(fontsize=9)
    ax.set_ylim(0, 100)

    # δ/h ratio vs force
    ax = axes[1, 0]
    F_range = np.linspace(0.01, 5, 200)
    delta_range = (3 * F_range * 1e-9 / (4 * E * np.sqrt(R)))**(2/3) * 1e9
    ratio_range = delta_range / h_cell_nm * 100
    ax.plot(F_range, ratio_range, 'b-', lw=2.5)
    ax.axhline(10, color='red', ls='--', lw=1.5, label='10% limit')
    ax.axvline(F_max_nN, color='gray', ls=':', lw=1)
    ax.fill_between(F_range, 0, 10, alpha=0.1, color='green')
    ax.fill_between(F_range, 10, np.max(ratio_range), alpha=0.1, color='red')
    ax.set_xlabel('Applied force (nN)')
    ax.set_ylabel('δ/h_cell (%)')
    ax.set_title('Indentation ratio vs force')
    ax.legend(fontsize=9)

    # Cantilever selection guide
    ax = axes[1, 1]
    k_range = np.logspace(-2, 1, 100)
    d_range = F_max_nN / k_range  # deflection at F_max
    ax.semilogx(k_range, d_range, 'b-', lw=2.5)
    ax.axhline(1, color='orange', ls='--', lw=1.5, label='Min. measurable (1 nm)')
    ax.axhline(50, color='red', ls='--', lw=1, label='Nonlinear (50 nm)')
    ax.axvline(k_cant, color='gray', ls=':', lw=1.5, label=f'k = {k_cant:.3f} N/m')
    ax.fill_between(k_range, 1, 50, alpha=0.1, color='green')
    ax.set_xlabel('Cantilever stiffness k (N/m)')
    ax.set_ylabel('Deflection at F_max (nm)')
    ax.set_title('Cantilever selection guide')
    ax.legend(fontsize=7)

    plt.tight_layout()
    plt.show()

interact(
    interactive_bacterial_stiffness,
    k_cant=FloatLogSlider(value=0.06, base=10, min=-2, max=1, step=0.1, description='k (N/m)'),
    F_max_nN=FloatSlider(value=1.0, min=0.1, max=10, step=0.1, description='F_max (nN)'),
    h_cell_nm=FloatSlider(value=800, min=200, max=2000, step=50, description='h_cell (nm)'),
    E_cell_kPa=FloatSlider(value=20, min=1, max=200, step=5, description='E* (kPa)'),
    R_nm=FloatSlider(value=25, min=5, max=100, step=5, description='R_tip (nm)')
);

---
## 10. Speed vs Information Trade-Off in Force Mapping

Different force mapping modes trade acquisition speed against information content (Section 5.7):

| Mode | Waveform | Speed | Information |
|------|----------|-------|-------------|
| **Force-Volume** | Full triangular ramp | Slow | Complete F-d curve |
| **QI™** | Compact approach–retraction | Medium | Key parameters |
| **PeakForce QNM** | Sinusoidal oscillation | Fast | Real-time properties |

Reducing the number of data points per curve speeds up acquisition but reduces the accuracy of parameter extraction.

**Key insight:** There is no universally "best" mode — the choice depends on whether you need full curve access (e.g., for model fitting) or rapid mapping of a few parameters.

### Tasks
1. Simulate force curves at different sampling densities.
2. Extract modulus from full vs reduced data and compare accuracy.
3. Estimate total acquisition time for each mode.

In [ ]:
def interactive_speed_info(n_points_full=512, n_points_fast=32, grid_pixels=64, ramp_rate_Hz=1):
    '''
    Compare force mapping modes: speed vs parameter accuracy.
    '''
    E_true = 25e3  # Pa
    R = 25e-9

    # Generate a single "true" force curve
    delta_full = np.linspace(0, 40, n_points_full)
    delta_m = delta_full * 1e-9
    F_full = (4/3) * E_true * np.sqrt(R) * delta_m**1.5 * 1e9

    rng = np.random.default_rng(42)
    noise = 0.05 * np.max(F_full)
    F_full_noisy = F_full + noise * rng.normal(size=len(delta_full))

    # Subsampled "fast" curve
    idx_fast = np.linspace(0, len(delta_full)-1, n_points_fast, dtype=int)
    delta_fast = delta_full[idx_fast]
    F_fast_noisy = F_full_noisy[idx_fast]

    # Fit both
    def hertz(d, E):
        return (4/3) * E * np.sqrt(R) * (d * 1e-9)**1.5 * 1e9

    try:
        E_fit_full, = curve_fit(hertz, delta_full, F_full_noisy, p0=[E_true])[0]
    except: E_fit_full = E_true
    try:
        E_fit_fast, = curve_fit(hertz, delta_fast, F_fast_noisy, p0=[E_true])[0]
    except: E_fit_fast = E_true

    # Timing estimates
    t_per_curve_full = n_points_full / ramp_rate_Hz / 1000  # seconds (simplified)
    t_per_curve_fast = n_points_fast / ramp_rate_Hz / 1000
    N_total = grid_pixels**2
    t_total_full = N_total * t_per_curve_full
    t_total_fast = N_total * t_per_curve_fast
    t_total_peak = N_total * 0.001  # PeakForce: ~1 ms per curve at 2 kHz

    # Run statistics: many curves
    E_fits_full = []
    E_fits_fast = []
    for trial in range(100):
        noise_trial = noise * np.random.default_rng(trial).normal(size=n_points_full)
        F_trial = F_full + noise_trial
        try:
            e, = curve_fit(hertz, delta_full, F_trial, p0=[E_true])[0]
            E_fits_full.append(e / 1e3)
        except: pass
        F_trial_fast = F_trial[idx_fast]
        try:
            e, = curve_fit(hertz, delta_fast, F_trial_fast, p0=[E_true])[0]
            E_fits_fast.append(e / 1e3)
        except: pass

    print(f"  Speed vs Information Trade-Off")
    print(f"  ─────────────────────────────────")
    print(f"  Grid: {grid_pixels}×{grid_pixels} = {N_total} curves")
    print()
    print(f"  {'Mode':<20s}  {'Points':>8s}  {'E* (kPa)':>10s}  {'Error':>8s}  {'Time':>12s}")
    print(f"  {'─'*20}  {'─'*8}  {'─'*10}  {'─'*8}  {'─'*12}")
    print(f"  {'Force-Volume':<20s}  {n_points_full:>8d}  {E_fit_full/1e3:>10.1f}  {(E_fit_full/E_true-1)*100:>+7.1f}%  {t_total_full:>10.1f} s")
    print(f"  {'QI / Fast':<20s}  {n_points_fast:>8d}  {E_fit_fast/1e3:>10.1f}  {(E_fit_fast/E_true-1)*100:>+7.1f}%  {t_total_fast:>10.1f} s")
    print(f"  {'PeakForce QNM':<20s}  {'~sinusoidal':>8s}  {'—':>10s}  {'—':>8s}  {t_total_peak:>10.1f} s")

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Full vs fast curves
    axes[0].scatter(delta_full, F_full_noisy, s=6, color='blue', alpha=0.3, label=f'Full ({n_points_full} pts)')
    axes[0].scatter(delta_fast, F_fast_noisy, s=30, color='red', zorder=3, label=f'Fast ({n_points_fast} pts)')
    axes[0].plot(delta_full, hertz(delta_full, E_fit_full), 'b-', lw=1.5, alpha=0.7)
    axes[0].plot(delta_full, hertz(delta_full, E_fit_fast), 'r--', lw=1.5, alpha=0.7)
    axes[0].set_xlabel('Indentation δ (nm)')
    axes[0].set_ylabel('Force (nN)')
    axes[0].set_title('Full vs reduced sampling')
    axes[0].legend(fontsize=8)

    # Histogram of fitted E* from many curves
    axes[1].hist(E_fits_full, bins=20, alpha=0.5, color='blue', label=f'Full: σ={np.std(E_fits_full):.1f} kPa')
    axes[1].hist(E_fits_fast, bins=20, alpha=0.5, color='red', label=f'Fast: σ={np.std(E_fits_fast):.1f} kPa')
    axes[1].axvline(E_true / 1e3, color='k', ls='--', lw=1.5, label='True E*')
    axes[1].set_xlabel('Fitted E* (kPa)')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Precision comparison (100 trials)')
    axes[1].legend(fontsize=8)

    # Acquisition time comparison
    modes = ['Force-\nVolume', 'QI /\nFast', 'PeakForce\nQNM']
    times = [t_total_full, t_total_fast, t_total_peak]
    colors = ['#2F6DB3', '#4C9A7D', '#C84A4A']
    axes[2].bar(modes, times, color=colors, edgecolor='white', width=0.5)
    for i, t in enumerate(times):
        if t > 60:
            axes[2].text(i, t + max(times)*0.02, f'{t/60:.1f} min', ha='center', fontsize=9)
        else:
            axes[2].text(i, t + max(times)*0.02, f'{t:.1f} s', ha='center', fontsize=9)
    axes[2].set_ylabel('Total acquisition time')
    axes[2].set_title(f'Time for {grid_pixels}×{grid_pixels} map')

    plt.tight_layout()
    plt.show()

interact(
    interactive_speed_info,
    n_points_full=IntSlider(value=512, min=64, max=2048, step=64, description='Full pts'),
    n_points_fast=IntSlider(value=32, min=8, max=128, step=8, description='Fast pts'),
    grid_pixels=IntSlider(value=64, min=16, max=256, step=16, description='Grid size'),
    ramp_rate_Hz=FloatSlider(value=1, min=0.1, max=10, step=0.1, description='Ramp rate (Hz)')
);

---

## Summary

These exercises covered the complete Chapter 5 workflow:

| Exercise | Topic | Key concept |
|----------|-------|-------------|
| 1 | F-d curve generation | Interaction regimes, hysteresis |
| 2 | Measurement chain | V → d → F, geometry correction |
| 3 | Hysteresis & dissipation | Trapezoidal integration, energy balance |
| 4 | Parameter extraction | F_adh, W_adh, k_eff, W_diss |
| 5 | Contact point detection | Threshold, RoV, error propagation |
| 6 | Hertz fitting | E* extraction, noise, fit range |
| 7 | Contact models | Hertz vs JKR vs DMT, Tabor parameter |
| 8 | Force mapping | Spatial parameter maps |
| 9 | Bacterial stiffness | Engineering constraints, 10% rule |
| 10 | Speed vs information | Sampling density, acquisition modes |

**Next:** Chapter 6 applies contact mechanics models in depth to extract intrinsic material properties from the force–indentation data prepared here.